# From Formulas to Modules

DeepLog parses formulas and, via the public `parse_formula_to_module` helper, lowers them with a {class}`~deeplog.formula.deeplogmodulefactory.deeplogmodulefactory.DeepLogModuleFactory`. The factory now ships defaults (equality/probability predicates and Klay circuits) so most notebooks don't need custom setup.

## Model Counting Warm-up

Consider two Boolean variables (`Burglary` and `Earthquake`) and a rule that fires whenever either event holds. We define the model count as a DeepLog formula and send it straight to `parse_formula_to_module`, which instantiates a factory with the built-in equality predicate and Klay circuit backend to produce a {class}`~deeplog.module.deeplog_module.DeepLogModule`.

In [ ]:
from deeplog import parse_formula_to_module


model_count_text = """
sum(Burglary): sum(Earthquake):
    =(Burglary,true)_boolean or =(Earthquake,true)_boolean
"""

model_count_module = parse_formula_to_module(model_count_text)
print("Model count: ", int(model_count_module()))

The module reports a model count of 3, matching the number of satisfying assignments (every valuation except the one where both burglary and earthquake are false).


## Weighted Model Counting

Weighted model counting augments each literal with a score. The default factory already registers [`ProbabilityPredicate`](deeplog.formula.predicates.builtin_predicates.ProbabilityPredicate), so we can keep using `parse_formula_to_module` with `_probability` leaves and let the factory wire the labels.

In [ ]:
weighted_formula_text = """
sum(Burglary): sum(Earthquake):
    (
        (=(Burglary,true)_boolean or =(Earthquake,true)_boolean)_probability
    )
    times
    (
        p(Burglary,0.8)_probability times p(Earthquake,0.3)_probability
    )
"""

weighted_module = parse_formula_to_module(weighted_formula_text)

print("Weighted model count:", float(weighted_module()))

## Expectation Operator

The same weighted model count can be computed more elegantly using the **expectation** operator. Given a boolean formula, the expectation compiles it into the probability semiring with determinism. The inputs are the probabilities of each atom being true, and the output is E[formula].

In [ ]:
expectation_text = """
expectation(Burglary, Earthquake):
    =(Burglary,true)_boolean or =(Earthquake,true)_boolean
"""

expectation_module = parse_formula_to_module(expectation_text)

import torch
# Input: P(Burglary=true)=0.8, P(Earthquake=true)=0.3
result = expectation_module(torch.tensor([[0.8, 0.3]]))
print("Expectation (WMC):", float(result))